# Learning quantum channels on 4×4 sudoku

This notebook trains actual Optyx quantum channels on the recurrent geometry proposed in [optyx#13](https://github.com/rel-int/optyx/issues/13). Sixteen cell channels exchange qubit messages with four row, four column and four square channels while retaining one private memory qubit each. One finite `at_time` unrolling is contracted with Cotengra on PyTorch tensors, and the Sudoku loss is differentiated through the contraction. There is no classical neural-network interpreter.

The documentation run is deliberately small. It trains four-way readouts for hidden cells on twelve puzzles, then reports per-cell accuracy and full-grid solve rate on four held-out puzzles alongside the earlier correct-versus-invalid ranking metric. This tests the quantum channel, recurrent routing, compressed contraction, and automatic differentiation together; it is not presented as a full 288-puzzle solver benchmark. PyTorch is required only for this learning experiment, not by the Optyx core package.

In [1]:
import time

import numpy as np
import torch
from cotengra import ReusableHyperCompressedOptimizer
from discopy import tensor

from optyx.channel import Channel, Diagram, qubit
from optyx.core.backends import DiscopyBackend
from optyx.core.contract import contract_tensor
from optyx.core.diagram import Box as CoreBox, bit as core_bit
from optyx.interaction import Box, CMap
from optyx.qubits import Ket

torch.set_num_threads(1)
_ = torch.manual_seed(7)

## Trainable Optyx channels

Cell maps are shared trainable isometries from seven to nine qubits: six message ports and one private memory qubit enter, then six messages, the updated memory, and two prediction qubits leave. We obtain the isometry by fixing two inputs of a nine-qubit real unitary to fresh zero ancillas. Constraint maps remain eight-qubit unitaries. A product of real rotations appears on either side of sixteen controlled rotations, keeping the cell parameter count (34) comparable to the constraint count (32). PyTorch is used because its reverse-mode gradient passes through Cotengra's compressed SVD contractions on this machine; the experimental JAX Metal backend fails the basic complex-array contraction.

In [2]:
size = 4
n_cells = size ** 2
n_constraints = 3 * size
cell_width = 9
constraint_width = 8
n_interactions = 16


def kron_all(arrays):
    result = arrays[0]
    for array in arrays[1:]:
        result = torch.kron(result, array)
    return result


def rotation_layer(parameters):
    matrices = []
    for theta in parameters:
        matrices.append(torch.stack((
            torch.stack((torch.cos(theta), -torch.sin(theta))),
            torch.stack((torch.sin(theta), torch.cos(theta))),
        )).to(torch.float64))
    return kron_all(matrices)


def channel_unitary(parameters, width):
    result = rotation_layer(parameters[:width])
    basis = torch.arange(2 ** width)
    interactions = parameters[2 * width:]
    for control in range(4):
        for offset, target in enumerate(range(width - 4, width)):
            theta = interactions[4 * control + offset]
            selected = (((basis >> control) & 1) == 1) & (
                ((basis >> target) & 1) == 0)
            lower = basis[selected]
            upper = lower ^ (1 << target)
            updated = result.clone()
            updated[lower] = (
                torch.cos(theta) * result[lower]
                - torch.sin(theta) * result[upper])
            updated[upper] = (
                torch.sin(theta) * result[lower]
                + torch.cos(theta) * result[upper])
            result = updated
    return rotation_layer(parameters[width:2 * width]) @ result


def trainable_channel(name, parameters, dom_width, cod_width):
    unitary = channel_unitary(parameters, cod_width)
    isometry = unitary[:2 ** dom_width]
    kraus = CoreBox(
        name, core_bit ** dom_width, core_bit ** cod_width,
        array=isometry)
    return Channel(
        name, kraus, qubit ** dom_width, qubit ** cod_width)


cell_parameters = (
    torch.randn(
        2 * cell_width + n_interactions, dtype=torch.float64) * .1
).requires_grad_()
constraint_parameters = (
    torch.randn(
        2 * constraint_width + n_interactions, dtype=torch.float64) * .1
).requires_grad_()

## The recurrent sudoku map

A cell reads and writes three message qubits, feeds one private memory qubit back to itself, and writes two prediction qubits to the environment. A constraint reads and writes four messages without private memory or prediction. The two directions give 96 edges and 192 paired feedback wires; the cells add 16 private wires, for 208 memory wires total. Every message port is paired, so the map has an empty boundary and 32 prediction qubits per tick.

In [3]:
def memberships(row, column):
    square = 2 * (row // 2) + column // 2
    position = 2 * (row % 2) + column % 2
    return (
        (n_cells + row, column),
        (n_cells + size + column, row),
        (n_cells + 2 * size + square, position),
    )


def sudoku_cmap():
    cell_channel = trainable_channel(
        "Cell", cell_parameters, 7, 9)
    constraint_channel = trainable_channel(
        "Constraint", constraint_parameters, 8, 8)
    boxes = [
        Box(f"cell_{index}", qubit ** 3, qubit ** 3, cell_channel,
            memory=qubit, prediction=qubit ** 2)
        for index in range(n_cells)
    ]
    boxes += [
        Box(f"{kind}_{index}", qubit ** 4, qubit ** 4,
            constraint_channel)
        for kind in ("row", "column", "square")
        for index in range(size)
    ]
    edges = []
    for cell in range(n_cells):
        row, column = divmod(cell, size)
        for slot, (constraint, position) in enumerate(
                memberships(row, column)):
            edges += [
                ((cell, slot), (constraint, size + position)),
                ((cell, 3 + slot), (constraint, position)),
            ]
    return CMap(boxes, edges)


sudoku = sudoku_cmap()
cell_isometry = sudoku.boxes[0].channel.kraus.array
assert torch.allclose(
    cell_isometry @ cell_isometry.T,
    torch.eye(2 ** 7, dtype=torch.float64), atol=1e-10)
topology = {
    "boxes": len(sudoku.boxes),
    "edges": len(sudoku.edges),
    "boundary_wires": len(sudoku.dom),
    "prediction_wires": len(sudoku.prediction),
    "paired_memory_wires": 2 * len(sudoku.edges),
    "internal_memory_wires": sum(
        len(box.memory) for box in sudoku.boxes),
    "memory_wires": len(sudoku.memory),
}
assert topology == {
    "boxes": 28, "edges": 96, "boundary_wires": 0,
    "prediction_wires": 32, "paired_memory_wires": 192,
    "internal_memory_wires": 16, "memory_wires": 208}
topology

{'boxes': 28,
 'edges': 96,
 'boundary_wires': 0,
 'prediction_wires': 32,
 'paired_memory_wires': 192,
 'internal_memory_wires': 16,
 'memory_wires': 208}

## A balanced dataset

We enumerate all 288 completed grids. Each selected puzzle hides cells 0 and 1 plus six others, and its eight remaining clues uniquely identify one grid in the complete corpus. The negative completion swaps the two hidden values in the first row, so it preserves every clue but violates sudoku constraints. Twelve solutions are used for training and four disjoint solutions for evaluation.

In [4]:
def sudoku_groups():
    rows = [tuple(row * size + column for column in range(size))
            for row in range(size)]
    columns = [tuple(row * size + column for row in range(size))
               for column in range(size)]
    squares = [
        tuple((2 * block_row + row) * size
              + 2 * block_column + column
              for row in range(2) for column in range(2))
        for block_row in range(2) for block_column in range(2)
    ]
    return rows + columns + squares


groups = sudoku_groups()
peers = [set() for _ in range(n_cells)]
for group in groups:
    for cell in group:
        peers[cell].update(set(group) - {cell})


def enumerate_solutions():
    result, grid = [], [0] * n_cells

    def visit(cell):
        if cell == n_cells:
            result.append(np.array(grid))
            return
        for value in range(1, size + 1):
            if all(grid[peer] != value for peer in peers[cell]):
                grid[cell] = value
                visit(cell + 1)
        grid[cell] = 0

    visit(0)
    return np.stack(result)


solutions = enumerate_solutions()
assert solutions.shape == (288, n_cells)

In [5]:
def puzzle_case(solution, random):
    available = np.arange(2, n_cells)
    while True:
        clue_cells = random.choice(
            available, n_cells // 2, replace=False)
        matches = np.all(
            solutions[:, clue_cells] == solution[clue_cells], axis=1)
        if matches.sum() == 1:
            clue = np.zeros(n_cells, dtype=int)
            clue[clue_cells] = solution[clue_cells]
            invalid = solution.copy()
            invalid[0], invalid[1] = invalid[1], invalid[0]
            return clue, solution, invalid


random = np.random.default_rng(19)
selection = random.choice(len(solutions), 16, replace=False)
cases = [puzzle_case(solutions[index], random) for index in selection]
train_cases, test_cases = cases[:12], cases[12:]
dataset = {
    "complete_corpus": len(solutions),
    "training_puzzles": len(train_cases),
    "held_out_puzzles": len(test_cases),
    "clues_per_puzzle": int(np.count_nonzero(train_cases[0][0])),
}
dataset

{'complete_corpus': 288,
 'training_puzzles': 12,
 'held_out_puzzles': 4,
 'clues_per_puzzle': 8}

## Efficient finite-time tensor semantics

We use the finite semantics of `Diagram.at_time`, with one unrolling (two recurrent ticks), rather than assuming that `fix` converges. Materialising its permutation on 240 qubit wires obscures the useful tensor-network structure, so `at_time_tensor_map` constructs the equivalent pure-Kraus Born-score network directly as a `discopy.tensor.CMap`. At tick zero every paired port starts in `|+⟩` and each private memory starts in `|0⟩`; later ticks connect partner outputs and each cell's memory output to its next input. Clue predictions are postselected on their digit at every tick, free predictions meet uniform effects until the last tick, and the final memory is closed with uniform effects. Thus recurrent routing is encoded by index labels rather than thousands of swap boxes.

In [6]:
zero = torch.tensor([1, 0], dtype=torch.float64)
one = torch.tensor([0, 1], dtype=torch.float64)
plus = torch.tensor([1, 1], dtype=torch.float64) / 2 ** .5


def digit_vectors(values):
    vectors = []
    for value in values:
        if value == 0:
            vectors += [plus, plus]
        else:
            value -= 1
            vectors += [
                one if value // 2 else zero,
                one if value % 2 else zero,
            ]
    return vectors


def at_time_tensor_map(cmap, clues, completion, unroll_steps):
    ticks = unroll_steps + 1
    boxes, pairs, next_port = [], [], 0

    def add(box):
        nonlocal next_port
        base = next_port
        next_port += len(box.dom) + len(box.cod)
        boxes.append(box)
        dom = list(range(base, base + len(box.dom)))
        cod = list(reversed(range(base + len(box.dom), next_port)))
        return dom, cod

    def state(value):
        box = tensor.Box(
            "state", tensor.Dim(), tensor.Dim(2), value)
        return add(box)[1][0]

    def effect(value):
        box = tensor.Box(
            "effect", tensor.Dim(2), tensor.Dim(), value)
        return add(box)[0][0]

    assert not cmap.boundary
    paired_inputs = {
        port: state(plus) for port in cmap.paired}
    private_inputs = {
        (box_index, memory_index): state(zero)
        for box_index, box in enumerate(cmap.boxes)
        for memory_index in range(len(box.memory))
    }
    with tensor.backend("pytorch"):
        local_tensors = {
            id(box.channel): tensor.Box(
                box.channel.name,
                tensor.Dim(2) ** len(box.channel.dom),
                tensor.Dim(2) ** len(box.channel.cod),
                box.channel.kraus.array)
            for box in cmap.boxes
        }
    local_inputs, local_outputs, predictions = [], [], []
    for _ in range(ticks):
        step_inputs, step_outputs, step_predictions = {}, {}, {}
        for box_index, box in enumerate(cmap.boxes):
            dom, cod = add(local_tensors[id(box.channel)])
            for port_index in range(len(box.ports)):
                port = box_index, port_index
                step_inputs["port", port] = dom[port_index]
                step_outputs["port", port] = cod[port_index]
            offset = len(box.ports)
            for memory_index in range(len(box.memory)):
                memory = box_index, memory_index
                step_inputs["memory", memory] = (
                    dom[offset + memory_index])
                step_outputs["memory", memory] = (
                    cod[offset + memory_index])
            offset += len(box.memory)
            step_predictions[box_index] = cod[
                offset:offset + len(box.prediction)]
        local_inputs.append(step_inputs)
        local_outputs.append(step_outputs)
        predictions.append(step_predictions)
    for step in range(ticks):
        for port in cmap.paired:
            source = paired_inputs[port] if step == 0 else (
                local_outputs[step - 1][
                    "port", cmap.partner[port]])
            pairs.append((
                source, local_inputs[step]["port", port]))
        for memory, source in private_inputs.items():
            if step:
                source = local_outputs[step - 1]["memory", memory]
            pairs.append((
                source, local_inputs[step]["memory", memory]))
        for cell in range(n_cells):
            value = clues[cell] or (
                completion[cell] if step == ticks - 1 else 0)
            for output, vector in zip(
                    predictions[step][cell], digit_vectors([value])):
                pairs.append((output, effect(vector)))
    for port in cmap.paired:
        pairs.append((
            local_outputs[-1]["port", cmap.partner[port]],
            effect(plus)))
    for memory in private_inputs:
        pairs.append((
            local_outputs[-1]["memory", memory], effect(plus)))
    edges = list(range(next_port))
    for left, right in pairs:
        edges[left], edges[right] = right, left
    assert all(edge != index for index, edge in enumerate(edges))
    return tensor.CMap(
        tensor.Dim(), tensor.Dim(), tuple(boxes), edges)

In [7]:
unroll_steps = 1
example_network = at_time_tensor_map(
    sudoku, train_cases[0][0], train_cases[0][1], unroll_steps)
network_summary = {
    "unroll_steps": unroll_steps,
    "ticks": unroll_steps + 1,
    "tensor_boxes": len(example_network.boxes),
    "tensor_ports": len(example_network.ports),
}
assert network_summary == {
    "unroll_steps": 1, "ticks": 2,
    "tensor_boxes": 536, "tensor_ports": 1376}
network_summary

{'unroll_steps': 1, 'ticks': 2, 'tensor_boxes': 536, 'tensor_ports': 1376}

## Born loss, contraction and backpropagation

The scalar contraction is a postselected amplitude for a final prediction. Its squared magnitude is a Born score. We retain the correct-versus-invalid ranking metric, and also obtain a four-way distribution for each hidden cell by postselecting that cell while leaving the other hidden predictions uniform. Cotengra reuses one compressed contraction path with bond dimension four, while PyTorch records every contraction and singular-value operation for reverse-mode differentiation.

In [8]:
path_optimizer = ReusableHyperCompressedOptimizer(
    chi=4, methods=("greedy-compressed",),
    max_repeats=1, optlib="random", seed=7,
    constants={"greedy-compressed": {"seed": 7}},
    progbar=False, parallel=False)


def completion_energy(clues, completion):
    network = at_time_tensor_map(
        sudoku_cmap(), clues, completion, unroll_steps)
    amplitude = contract_tensor(
        network, backend="pytorch", optimize=path_optimizer,
        max_bond=4, cutoff=1e-8, dtype=float).array
    probability = amplitude.abs().square().real
    return -torch.log(probability.clamp_min(
        torch.finfo(torch.float64).tiny))


def correct_probabilities(dataset):
    probabilities = []
    for clues, solution, invalid in dataset:
        energies = torch.stack((
            completion_energy(clues, solution),
            completion_energy(clues, invalid),
        ))
        probabilities.append(torch.softmax(-energies, 0)[0].item())
    return probabilities


def cell_distribution(clues, cell):
    energies = []
    for digit in range(1, size + 1):
        completion = np.zeros(n_cells, dtype=int)
        completion[cell] = digit
        energies.append(completion_energy(clues, completion))
    return torch.softmax(-torch.stack(energies), 0)


def decode(dataset):
    n_correct, n_hidden, n_solved = 0, 0, 0
    grids = []
    for clues, solution, _ in dataset:
        prediction = clues.copy()
        for cell in np.flatnonzero(clues == 0):
            probabilities = cell_distribution(clues, cell)
            prediction[cell] = probabilities.argmax().item() + 1
            n_correct += prediction[cell] == solution[cell]
            n_hidden += 1
        n_solved += np.array_equal(prediction, solution)
        grids.append(prediction.reshape(size, size).tolist())
    return {
        "per_cell_accuracy": n_correct / n_hidden,
        "full_grid_solve_rate": n_solved / len(dataset),
        "grids": grids,
    }


initial_test = correct_probabilities(test_cases)
initial_readout = decode(test_cases)
{
    "candidate_probabilities": initial_test,
    "per_cell_accuracy": initial_readout["per_cell_accuracy"],
    "full_grid_solve_rate": (
        initial_readout["full_grid_solve_rate"]),
}

{'candidate_probabilities': [0.5032472838322877,
  0.49669104315553253,
  0.42818515494796483,
  0.5653621371936859],
 'per_cell_accuracy': 0.21875,
 'full_grid_solve_rate': 0.0}

In [9]:
learning_optimizer = torch.optim.Adam(
    (cell_parameters, constraint_parameters), lr=0.03)
history = []
gradient_norm = None
started = time.perf_counter()
for step in range(24):
    clues, solution, _ = train_cases[step % len(train_cases)]
    hidden = np.flatnonzero(clues == 0)
    cell = hidden[(step // len(train_cases)) % len(hidden)]
    learning_optimizer.zero_grad()
    energies = []
    for digit in range(1, size + 1):
        completion = np.zeros(n_cells, dtype=int)
        completion[cell] = digit
        energies.append(completion_energy(clues, completion))
    energies = torch.stack(energies)
    target = solution[cell] - 1
    loss = energies[target] + torch.logsumexp(-energies, 0)
    loss.backward()
    if step == 0:
        gradient_norm = torch.sqrt(sum(
            parameter.grad.square().sum()
            for parameter in (
                cell_parameters, constraint_parameters))).item()
    learning_optimizer.step()
    history.append(loss.item())

final_test = correct_probabilities(test_cases)
final_readout = decode(test_cases)
elapsed = time.perf_counter() - started
results = {
    "initial_gradient_norm": gradient_norm,
    "training_updates": len(history),
    "first_twelve_loss": float(np.mean(history[:12])),
    "last_twelve_loss": float(np.mean(history[-12:])),
    "initial_test_probability": float(np.mean(initial_test)),
    "final_test_probability": float(np.mean(final_test)),
    "initial_candidate_accuracy": float(
        np.mean(np.asarray(initial_test) > .5)),
    "final_candidate_accuracy": float(
        np.mean(np.asarray(final_test) > .5)),
    "initial_per_cell_accuracy": (
        initial_readout["per_cell_accuracy"]),
    "final_per_cell_accuracy": (
        final_readout["per_cell_accuracy"]),
    "initial_full_grid_solve_rate": (
        initial_readout["full_grid_solve_rate"]),
    "final_full_grid_solve_rate": (
        final_readout["full_grid_solve_rate"]),
    "training_seconds": elapsed,
}
assert results["initial_gradient_norm"] > 1e-6
assert (results["final_test_probability"]
        > results["initial_test_probability"])
assert (results["final_per_cell_accuracy"]
        > results["initial_per_cell_accuracy"])
results

{'initial_gradient_norm': 6.909175314929146,
 'training_updates': 24,
 'first_twelve_loss': 1.5580121059890832,
 'last_twelve_loss': 1.9574421546586542,
 'initial_test_probability': 0.4983714047823677,
 'final_test_probability': 0.5020030981279157,
 'initial_candidate_accuracy': 0.5,
 'final_candidate_accuracy': 0.5,
 'initial_per_cell_accuracy': 0.21875,
 'final_per_cell_accuracy': 0.34375,
 'initial_full_grid_solve_rate': 0.0,
 'final_full_grid_solve_rate': 0.0,
 'training_seconds': 45.948721834109165}

In [10]:
example = {
    "clues": test_cases[0][0].reshape(size, size).tolist(),
    "correct": test_cases[0][1].reshape(size, size).tolist(),
    "invalid": test_cases[0][2].reshape(size, size).tolist(),
    "prediction": final_readout["grids"][0],
    "correct_probability": final_test[0],
}
example

{'clues': [[0, 0, 0, 0], [0, 1, 0, 3], [4, 3, 2, 1], [0, 2, 0, 4]],
 'correct': [[3, 4, 1, 2], [2, 1, 4, 3], [4, 3, 2, 1], [1, 2, 3, 4]],
 'invalid': [[4, 3, 1, 2], [2, 1, 4, 3], [4, 3, 2, 1], [1, 2, 3, 4]],
 'prediction': [[3, 1, 3, 2], [1, 1, 1, 3], [4, 3, 2, 1], [1, 2, 1, 4]],
 'correct_probability': 0.4970641922590678}

## Finite `at_time` semantics

One unrolling gives two recurrent ticks, enough for a message to travel from a cell to a constraint and back. We deliberately do not call `fix`: convergence is not established for the learned channels. The executable probe below uses the public `at_time(1)` API on a recurrent wire with an explicit memory state; the full Sudoku contraction above is its direct combinatorial-map representation.

In [11]:
wire = Box("wire", qubit, qubit ** 2, Diagram.id(qubit ** 3))
probe = CMap([wire], [((0, 1), (0, 2))])
process = (
    Ket(1) @ Diagram.id(probe.memory) >> probe.step
).feedback(
    dom=probe.dom[:0], cod=probe.cod, mem=probe.memory,
    state=Ket(0) @ Ket(0))
late = process.at_time(unroll_steps).eval(DiscopyBackend())
assert np.allclose(late.density_matrix, [[0, 0], [0, 1]])

This experiment establishes the end-to-end path that was missing: Optyx channel parameters → finite recurrent quantum tensor map → Cotengra compression → PyTorch loss → channel gradients → per-cell Sudoku predictions. The small run improves held-out cell accuracy but does not yet solve a complete held-out grid. Batching contractions, memory ablation, and comparing more unroll depths and bond dimensions remain explicit benchmark work rather than hidden assumptions in this documentation run.